# Global Step 1: Dataset Ingestion from Google Drive (`gdown`) & AWS S3 Sync

**RUN ONCE:** This notebook downloads the dataset directly from Google Drive using `gdown`, extracts it locally to `dataset/`, deletes the zip file to free up disk space on the 16 GB EBS limit, and syncs the dataset to your Amazon S3 bucket.

Once executed, all models and training notebooks can re-use the dataset from disk or S3 without downloading it again.

In [ ]:
# 1. Install required packages on SageMaker
!pip install -q gdown boto3 pandas tqdm unidecode indic-transliteration

In [ ]:
import os
import sys
import zipfile
import gdown
import boto3

# Base directory configuration
base_dir = os.path.dirname(os.path.dirname(os.path.abspath("")))
dataset_dir = os.path.join(base_dir, "dataset")
s3_bucket = "amazon-ml-challenge-2026-entity-resolution"
s3_prefix = "dataset"

print(f"Base Directory   : {base_dir}")
print(f"Dataset Directory: {dataset_dir}")
print(f"S3 Target Bucket : s3://{s3_bucket}/{s3_prefix}/")

## 2. Download Dataset from Google Drive
Paste your Google Drive shareable file link or File ID below.

In [ ]:
# PASTE YOUR GOOGLE DRIVE SHAREABLE LINK OR FILE ID HERE
GDRIVE_FILE_ID = "1G8EJe2gARnLJXj12GzWf5IR-QD364Fnp"  # Replace with your actual Drive File ID or URL
GDRIVE_URL = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"

download_zip_path = os.path.join(base_dir, "dataset.zip")

print("Downloading dataset from Google Drive via gdown...")
if not os.path.exists(dataset_dir) or not os.path.exists(os.path.join(dataset_dir, "train")):
    gdown.download(GDRIVE_URL, download_zip_path, quiet=False)
    
    if os.path.exists(download_zip_path) and zipfile.is_zipfile(download_zip_path):
        print(f"Extracting {download_zip_path} to {dataset_dir}...")
        with zipfile.ZipFile(download_zip_path, 'r') as zip_ref:
            zip_ref.extractall(base_dir)
        print("Dataset successfully extracted!")
        
        # Remove zip file to free up ~3 GB disk space on 16 GB EBS volume
        if os.path.exists(download_zip_path):
            os.remove(download_zip_path)
            print("Successfully deleted dataset.zip to save disk space.")
    else:
        print("Downloading folder directly from Google Drive...")
        gdown.download_folder(url="https://drive.google.com/drive/u/0/folders/1G8EJe2gARnLJXj12GzWf5IR-QD364Fnp", output=dataset_dir)
else:
    print(f"Dataset already exists at {dataset_dir} — no re-download needed!")

## 3. Sync Dataset to Amazon S3 Bucket
Uploads raw dataset files (`train_source1/2/3.tsv` and `test_source1/2/3.tsv`) to Amazon S3.

In [ ]:
def sync_directory_to_s3(local_dir: str, bucket: str, prefix: str) -> None:
    s3_client = boto3.client("s3")
    for root, _, files in os.walk(local_dir):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_dir)
            s3_key = os.path.join(prefix, relative_path).replace("\\", "/")
            try:
                s3_client.upload_file(local_path, bucket, s3_key)
                print(f"Uploaded {relative_path} -> s3://{bucket}/{s3_key}")
            except Exception as e:
                print(f"Failed to upload {local_path}: {e}")
    print(f"\n--> Successfully synced {local_dir} to s3://{bucket}/{prefix}/")

if os.path.exists(dataset_dir):
    sync_directory_to_s3(local_dir=dataset_dir, bucket=s3_bucket, prefix=s3_prefix)
else:
    print(f"Dataset directory not found at {dataset_dir}")